# Fase 4 — Análise dos 5 desafios principais

Base: `fauna.monitoramento.gold_registros`. Cada desafio vira uma seção:
pergunta, consulta, e conclusão escrita a partir do resultado — não custa
lembrar: a conclusão só é validada depois de olhar o número, nunca antes.

In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG fauna")
spark.sql("USE SCHEMA monitoramento")

df_gold = spark.table("fauna.monitoramento.gold_registros")
print(f"Base carregada: {df_gold.count()} registros")

## Desafio 1 — Animais solitários ou em grupo?

**Pergunta:** Qual o número médio de indivíduos por registro de capivara? A
maioria dos registros envolve indivíduos isolados ou grupos?

In [0]:
df_capivara = df_gold.filter(F.col("especie") == "Capivara")

print(f"Total de registros de capivara: {df_capivara.count()}")

df_capivara.select(
    F.mean("individuos").alias("media_individuos"),
    F.expr("percentile_approx(individuos, 0.5)").alias("mediana_individuos"),
    F.min("individuos").alias("minimo"),
    F.max("individuos").alias("maximo"),
).show()

In [0]:
df_capivara.withColumn(
    "tipo_registro", F.when(F.col("individuos") == 1, "Solitário").otherwise("Grupo (2+)")
).groupBy("tipo_registro").count().withColumn(
    "percentual", F.round(F.col("count") / df_capivara.count() * 100, 1)
).show()

### Visualização

In [0]:
display(
    df_capivara.groupBy("individuos").count().orderBy("individuos")
)

Databricks visualization. Run in Databricks to view.

### Conclusão — Desafio 1

As capivaras são registradas predominantemente **em grupo**: 99,2% dos 731
registros envolvem 2 ou mais indivíduos, com média de ~6 indivíduos por
registro (mediana também 6, e máximo de 14). Registros solitários são raros
(0,8%, apenas 6 casos). Isso é consistente com o comportamento social
conhecido da espécie, que costuma viver em grupos familiares.

## Desafio 2 — Quando a fauna está mais ativa?

**Pergunta:** Em quais períodos do dia há mais registros de animais? Escolha
três espécies e compare seus horários de maior ocorrência.

**Espécies escolhidas:** Onça-pintada, Lobo-guará e Capivara — uma de cada
"papel" ecológico: predador de topo, predador menor, e presa/herbívoro.

In [0]:
print("Distribuição geral por período do dia:")
df_gold.groupBy("periodo_dia").count().orderBy(F.desc("count")).show()

In [0]:
especies_selecionadas = ["Onça-pintada", "Lobo-guará", "Capivara"]

df_comparacao = (
    df_gold
    .filter(F.col("especie").isin(especies_selecionadas))
    .groupBy("especie", "periodo_dia")
    .count()
)

display(df_comparacao.orderBy("especie", "periodo_dia"))

### Normalizar por espécie (percentual, não contagem bruta)

In [0]:
from pyspark.sql.window import Window

total_por_especie = Window.partitionBy("especie")

df_comparacao_pct = (
    df_comparacao
    .withColumn("total_especie", F.sum("count").over(total_por_especie))
    .withColumn("percentual", F.round(F.col("count") / F.col("total_especie") * 100, 1))
    .orderBy("especie", F.desc("percentual"))
)

df_comparacao_pct.select("especie", "periodo_dia", "count", "percentual").show(20, truncate=False)

### Conclusão — Desafio 2

No geral, a atividade da fauna é maior pela manhã (1.419 registros) e à tarde
(1.317), caindo à noite (1.177) e de madrugada (1.087) — mas esse padrão
geral esconde diferenças grandes entre espécies.

Comparando as três espécies escolhidas: a **capivara** é fortemente diurna
(78,6% dos registros entre manhã e tarde), o **lobo-guará** tem atividade mais
distribuída ao longo do dia, sem período dominante, e a **onça-pintada** é
majoritariamente noturna/crepuscular (80% dos registros entre noite e
madrugada, apenas 5% à tarde). Isso mostra que o padrão de atividade "da
fauna em geral" não se aplica igualmente a cada espécie — cada uma ocupa um
nicho temporal diferente.

## Desafio 3 — Vale a pena procurar anfíbios em períodos mais úmidos?

**Pergunta:** Os anfíbios são registrados com maior frequência quando a
umidade é mais alta? Compare a quantidade de registros em diferentes níveis
de umidade.

In [0]:
df_anfibios = df_gold.filter(F.col("grupo") == "Anfíbio")

total_anfibios = df_anfibios.count()
sem_umidade = df_anfibios.filter(F.col("_flag_umidade_ausente")).count()

print(f"Total de registros de anfíbios: {total_anfibios}")
print(f"Sem leitura de umidade (sensor falhou): {sem_umidade} ({sem_umidade/total_anfibios*100:.1f}%)")

In [0]:
df_anfibios_com_umidade = df_anfibios.filter(~F.col("_flag_umidade_ausente"))

df_anfibios_categorizado = df_anfibios_com_umidade.withColumn(
    "categoria_umidade",
    F.when(F.col("umidade_pct") < 60, "Baixa (<60%)")
     .when(F.col("umidade_pct") <= 80, "Média (60-80%)")
     .otherwise("Alta (>80%)")
)

total_valido = df_anfibios_com_umidade.count()

df_anfibios_categorizado.groupBy("categoria_umidade").count() \
    .withColumn("percentual", F.round(F.col("count") / total_valido * 100, 1)) \
    .orderBy("categoria_umidade") \
    .show()

### Comparar com a distribuição geral (fora dos anfíbios)

In [0]:
df_outros_com_umidade = (
    df_gold
    .filter((F.col("grupo") != "Anfíbio") & (~F.col("_flag_umidade_ausente")))
)

total_outros = df_outros_com_umidade.count()

df_outros_categorizado = df_outros_com_umidade.withColumn(
    "categoria_umidade",
    F.when(F.col("umidade_pct") < 60, "Baixa (<60%)")
     .when(F.col("umidade_pct") <= 80, "Média (60-80%)")
     .otherwise("Alta (>80%)")
)

df_outros_categorizado.groupBy("categoria_umidade").count() \
    .withColumn("percentual", F.round(F.col("count") / total_outros * 100, 1)) \
    .orderBy("categoria_umidade") \
    .show()

### Conclusão — Desafio 3

Sim, os anfíbios são registrados com muito mais frequência em períodos de
umidade alta. 65,8% dos registros de anfíbios (com leitura de sensor válida)
ocorrem com umidade acima de 80%, contra apenas 33,9% para os demais grupos
faunísticos no mesmo período — quase o dobro da proporção. No outro extremo,
só 1,7% dos anfíbios aparecem em umidade baixa (<60%), contra 12,4% dos
outros grupos. Essa comparação com o baseline confirma que não é um efeito
geral do clima do parque, e sim um padrão específico: **vale a pena
concentrar o monitoramento de anfíbios em janelas de alta umidade**. (71
registros de anfíbios, 5,6% do total, ficaram sem leitura de umidade por
falha de sensor e foram excluídos desta análise.)

## Desafio 4 — Onde concentrar o monitoramento?

**Pergunta:** Quais câmeras apresentam maior quantidade de registros e maior
variedade de espécies? Essas câmeras estão concentradas em alguma área?

In [0]:
df_camera_stats = (
    df_gold
    .groupBy("id_camera", "area")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("especie").alias("variedade_especies"),
    )
    .orderBy(F.desc("total_registros"))
)

display(df_camera_stats)

### Agregar por área

In [0]:
df_area_stats = (
    df_gold
    .groupBy("area")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("especie").alias("variedade_especies"),
        F.countDistinct("id_camera").alias("num_cameras"),
    )
    .withColumn("media_por_camera", F.round(F.col("total_registros") / F.col("num_cameras"), 1))
    .orderBy(F.desc("total_registros"))
)

df_area_stats.show(truncate=False)

### Conclusão — Desafio 4

As câmeras mais ativas são CAM09 (523), CAM12 (504) e CAM10 (474) — três das
quatro câmeras de **Veredas e áreas úmidas**, que concentra a maior atividade
entre as três áreas (1.817 registros, média de 454,3 por câmera, contra 406,0
no Cerrado aberto e 389,8 na Mata de galeria).

A variedade de espécies, por outro lado, quase não diferencia as câmeras —
praticamente todas registram 12 ou 13 das 13 espécies do parque. A única
exceção notável é o Cerrado aberto, que fica em 12 espécies (uma a menos que
as outras duas áreas), sugerindo que uma espécie específica evita esse
ambiente mais aberto.

**Recomendação:** concentrar o próximo período de monitoramento em Veredas e
áreas úmidas, sem abrir mão de manter cobertura nas outras áreas — a
diversidade de espécies já está bem distribuída, o ganho estaria em captar
mais volume de atividade.

## Desafio 5 — Onde procurar a onça-pintada?

**Pergunta:** Com base nos registros deste mês, em quais câmeras, áreas e
períodos do dia você recomendaria concentrar o próximo monitoramento?

**Nota metodológica:** onça-pintada tem apenas 20 registros no total — baixo
o suficiente para exigir cautela ao generalizar por câmera individual.
Priorizamos agregações por área e período, mais robustas com essa amostra.

In [0]:
df_onca = df_gold.filter(F.col("especie") == "Onça-pintada")
total_onca = df_onca.count()
print(f"Total de registros de onça-pintada: {total_onca}")

print("\nPor câmera:")
df_onca.groupBy("id_camera", "area").count().orderBy(F.desc("count")).show(truncate=False)

print("Por área (mais robusto que por câmera, com essa amostra):")
df_onca.groupBy("area").count() \
    .withColumn("percentual", F.round(F.col("count") / total_onca * 100, 1)) \
    .orderBy(F.desc("count")).show(truncate=False)

print("Por período do dia:")
df_onca.groupBy("periodo_dia").count() \
    .withColumn("percentual", F.round(F.col("count") / total_onca * 100, 1)) \
    .orderBy(F.desc("count")).show(truncate=False)

### Conclusão e recomendação — Desafio 5

Combinando os dois resultados robustos desta análise (área, com 20 registros
distribuídos de forma bem desigual entre as três áreas) e o período do dia
(já visto no Desafio 2, onde a onça-pintada mostrou ser majoritariamente
noturna/crepuscular):

- **Área:** Mata de galeria concentra 85% dos registros (17 de 20) — nenhum
  registro em Cerrado aberto, área aberta que não oferece a cobertura vegetal
  que a espécie busca para caçar.
- **Câmeras:** dentro da Mata de galeria, CAM06 (6 registros) e CAM08 (5)
  lideram, mas com uma amostra de 20 registros no total, evitamos recomendar
  uma única câmera isolada — o sinal mais confiável está no nível de área.
- **Período do dia:** Noite (50%) e Madrugada (30%) somam 80% dos registros —
  praticamente o oposto do padrão diurno da capivara visto no Desafio 2.

**Recomendação:** concentrar o próximo monitoramento nas câmeras de **Mata de
galeria**, priorizando os turnos de **noite e madrugada**. Vale reforçar: com
apenas 20 registros no mês, essa recomendação deve ser tratada como um
indicativo inicial, não uma certeza estatística — mais meses de dados
fortaleceriam a conclusão.